# BirdCLEF+ 2025 - Pretrained models, A3: Bird-MAE-Base embeddings + linear probe (GPU)

This notebook uses a frozen `DBD-research-group/Bird-MAE-Base` as a feature extractor (no fine-tuning) and trains our own linear probe on top. The prototypical probe is not part of this build. The protocol is the same as for A2, so the two are comparable.

**Model:** fetched at a fixed Hugging Face commit, checksum verified, and loaded with `AutoModel` / `AutoFeatureExtractor` and `trust_remote_code=True`. The output is 768-d mean-pooled patch tokens.

**Input:** the repo's own Kaldi fbank, because the baseline mel cache cannot be used here. 32 kHz mono, one 5 s window per call.

**Embedding:** clips cut to 30 s, non-overlapping 5 s windows. A clip shorter than 5 s gives one zero-padded window, and a trailing part shorter than 5 s is dropped.

**Labels:** every window inherits its clip's labels (primary 1.0, secondary 0.5). Windows are weighted by 1 / (windows in the clip), so each clip counts once per epoch.

**Split:** our team's baseline split (stratified 80/20, seed 42) on its `train_index.csv`, on clips.

**Validation:** the baseline protocol (first 5 s of each hold-out clip, primary label only, macro ROC-AUC skipping classes with no positives) picks the epoch. We also report max-over-windows per clip.

In [ ]:
import os
import torch

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE_TEST:
    print("SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import subprocess, sys
from pathlib import Path

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
# config.json of the pinned commit records transformers 4.38.0, the only version the repo documents.
# Installed to a separate folder so the image's own packages are left alone. The inference notebook
# installs the same three wheels, offline, from this notebook's output.
PINS = {"transformers": "4.38.0", "tokenizers": "0.15.2", "huggingface_hub": "0.21.4"}
PYDEPS = Path(os.environ.get("M3_PYDEPS", "/tmp/bmae_pydeps"))

if not SMOKE_TEST:
    wheels = OUT / "bmae_wheels"; wheels.mkdir(exist_ok=True)
    pins = [f"{k}=={v}" for k, v in PINS.items()]
    this_py = f"{sys.version_info.major}.{sys.version_info.minor}"
    for py in ["3.10", "3.11", "3.12", "3.13"]:
        r = subprocess.run([sys.executable, "-m", "pip", "download", "-q", *pins, "--no-deps", "--only-binary=:all:",
                            "--python-version", py, "--implementation", "cp", "--platform", "manylinux2014_x86_64",
                            "--platform", "manylinux_2_17_x86_64", "-d", str(wheels)], capture_output=True, text=True)
        if r.returncode != 0 and py == this_py:
            raise RuntimeError(f"Pinned wheels unavailable for this Python ({py}): {r.stderr[-400:]}")
    print("Wheels:", sorted(p.name for p in wheels.glob("*.whl")))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", str(PYDEPS),
                    "--find-links", str(wheels), *pins], check=True)
sys.path.insert(0, str(PYDEPS))

In [ ]:
import ast, glob, hashlib, json, time, urllib.request
import numpy as np
import pandas as pd
import soundfile as sf
import torch.nn as nn
from scipy.signal import resample_poly
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import transformers
from transformers import AutoFeatureExtractor, AutoModel

print("transformers", transformers.__version__, "from", transformers.__file__)
assert Path(transformers.__file__).is_relative_to(PYDEPS), "pinned transformers is not the one imported"
if not SMOKE_TEST and transformers.__version__ != PINS["transformers"]:
    raise RuntimeError(f"Expected transformers {PINS['transformers']}, got {transformers.__version__}")
if not SMOKE_TEST:
    import importlib.metadata
    print("torch", torch.__version__, "| torchaudio", importlib.metadata.version("torchaudio"))

CFG = dict(max_seconds=30, val_frac=0.2, seed=42, secondary_weight=0.5,
           loader_clips=16, embed_batch=128, epochs=5 if SMOKE_TEST else 30, batch_size=1024, lr=1e-3,
           weight_decay=1e-2, preprocess_version=1)
NW = int(os.environ.get("M3_WORKERS", "0" if SMOKE_TEST else "4"))   # audio + fbank workers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])

def find_dir(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
MELS = find_dir("train_index.csv", "M3_MELS_DIR", "../../outputs/jamie_make_mels")
print("Data:", DATA, "| M1 index:", MELS, "| device:", DEVICE)

## Clips, labels and the baseline's exact split

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

if "filename" in index.columns:
    filenames = index["filename"].astype(str).tolist()
else:
    # Only safe when M1's index is row-for-row train.csv; anything else must stop here.
    train_csv = pd.read_csv(DATA / "train.csv")
    same = len(train_csv) == len(index) and (train_csv["primary_label"].astype(str).values
                                             == index["primary_label"].astype(str).values).all()
    if not same:
        raise RuntimeError(f"train_index.csv has no filename column and does not align with train.csv; "
                           f"columns: {list(index.columns)}")
    filenames = train_csv["filename"].astype(str).tolist()
    print("train_index.csv has no filename column: aligned to train.csv row by row")
files = [DATA / "train_audio" / f for f in filenames]

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

# Copied from M1 train-v2: same call, same array of clip numbers, same stratify, same seed.
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va))
pd.DataFrame({"clip": idx_va, "filename": [filenames[i] for i in idx_va]}).to_csv(OUT / "val_clips.csv", index=False)

## Extract embeddings once (reused from disk when already present)

In [ ]:
SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, Bird-MAE's native clip length
EMB_DIM = 768
HF_REPO = "DBD-research-group/Bird-MAE-Base"
HF_REVISION = "6cc416d1a7ae2af29b6b866499b3b047a8f01304"
BIRDMAE_SHA256 = "402103ecc81787ca6317ad78377fb38a912b3548f23a18f424d7a0dc31f754c5"   # model.safetensors
MODEL_FILES = ["model.safetensors", "config.json", "preprocessor_config.json",
               "configuration_bird_mae.py", "feature_extractor.py", "modeling_bird_mae.py"]

def to_32k_mono(y, sr):
    """Shared with the other A3 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)
    return y.astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Same as A2; shared with the other A3 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
def load_clip_windows(path):
    with sf.SoundFile(str(path)) as f:
        sr = f.samplerate
        y = f.read(min(len(f), CFG["max_seconds"] * sr), dtype="float32", always_2d=True)
    y = to_32k_mono(y, sr)
    n_full = len(y) // WIN
    if n_full == 0:
        y = np.pad(y, (0, WIN - len(y)))
        n_full = 1
    return np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_full)])

def windows_to_specs(w):
    """One feature-extractor call per window, never a batched call: the repo pads the ~14 missing frames
    with the minimum of whatever batch it is given, so a batched call would make a window's input depend
    on its neighbours and differ between training and inference."""
    return torch.cat([fe(x) for x in w]).float()      # [n, 1, 512, 128]

class ClipDataset(torch.utils.data.Dataset):
    def __len__(self):
        return len(files)
    def __getitem__(self, j):
        try:
            return j, windows_to_specs(load_clip_windows(files[j])), ""
        except Exception as e:             # counted and limited below
            return j, torch.zeros((0, 1, 512, 128)), repr(e)

def collate(items):
    specs = torch.cat([it[1] for it in items])
    clips = torch.tensor([it[0] for it in items for _ in range(len(it[1]))], dtype=torch.long)
    wins = torch.tensor([k for it in items for k in range(len(it[1]))], dtype=torch.long)
    return specs, clips, wins, [(it[0], it[2]) for it in items if it[2]]

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

def get_model_dir():
    """Every file comes from the pinned commit. Loading from a local folder is also what inference does."""
    if os.environ.get("M3_BIRDMAE_DIR"):
        return Path(os.environ["M3_BIRDMAE_DIR"])
    target = Path("/tmp/bird_mae_base")        # outside /kaggle/working, keeps 342 MB out of the output
    target.mkdir(exist_ok=True)
    t0 = time.time()
    for name in MODEL_FILES:
        if not (target / name).exists():
            urllib.request.urlretrieve(f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{name}", target / name)
    print(f"model files ready in {time.time() - t0:.0f}s")
    got = sha256_of(target / "model.safetensors")
    if got != BIRDMAE_SHA256:
        raise RuntimeError(f"model.safetensors checksum mismatch: {got}")
    return target

def fingerprint():
    return dict(filenames_sha=hashlib.sha256("\n".join(filenames).encode()).hexdigest(),
                hf_revision=HF_REVISION, birdmae_sha=BIRDMAE_SHA256, transformers=PINS["transformers"],
                max_seconds=CFG["max_seconds"], preprocess_version=CFG["preprocess_version"])

def find_cache():
    want = fingerprint()
    for d in [OUT] + [Path(p).parent for p in glob.glob("/kaggle/input/**/birdmae_embeddings.npy", recursive=True)]:
        meta = d / "birdmae_cache_meta.json"
        if meta.exists() and (d / "birdmae_embeddings.npy").exists() and (d / "birdmae_windows.csv").exists():
            if json.load(open(meta)).get("fingerprint") == want:
                return d
    return None

cache = find_cache()
if cache is None:
    model_dir = get_model_dir()
    file_shas = {n: sha256_of(model_dir / n) for n in MODEL_FILES}
    fe = AutoFeatureExtractor.from_pretrained(str(model_dir), trust_remote_code=True)
    bird_mae, info = AutoModel.from_pretrained(str(model_dir), trust_remote_code=True, output_loading_info=True)
    if info["missing_keys"] or info["unexpected_keys"]:
        raise RuntimeError(f"Weights did not load cleanly: {info['missing_keys'][:5]} {info['unexpected_keys'][:5]}")
    assert bird_mae.config.global_pool == "mean", bird_mae.config.global_pool
    bird_mae.eval().requires_grad_(False).to(DEVICE)     # frozen: never trained here
    use_amp = DEVICE == "cuda"

    # Fails in the first minute if the audio path, feature extractor or model do not fit together.
    probe = windows_to_specs(load_clip_windows(files[0]))
    assert probe.shape[1:] == (1, 512, 128), probe.shape
    with torch.inference_mode(), torch.autocast(DEVICE, enabled=use_amp):
        out = bird_mae(probe.to(DEVICE))[0]
    assert out.shape == (len(probe), EMB_DIM), out.shape
    print("probe OK:", probe.shape, "->", out.shape)

    loader = torch.utils.data.DataLoader(
        ClipDataset(), batch_size=CFG["loader_clips"], shuffle=False, num_workers=NW, collate_fn=collate,
        pin_memory=DEVICE == "cuda", prefetch_factor=4 if NW else None)
    BATCH = CFG["embed_batch"]
    embs, clip_ids, win_ids, failed = [], [], [], []
    n_done = 0
    t_start = time.time(); t_gpu = 0.0
    for step, (specs, clips, wins, errs) in enumerate(loader):
        failed += [(filenames[j], e) for j, e in errs]
        seen = (step + 1) * CFG["loader_clips"]
        if len(failed) > max(20, 0.05 * seen):     # stop early rather than spend GPU time on a broken path
            raise RuntimeError(f"{len(failed)} of {seen} clips failed, first: {failed[:3]}")
        t = time.time()
        for i in range(0, len(specs), BATCH):
            x = specs[i:i + BATCH].to(DEVICE, non_blocking=True)
            with torch.inference_mode(), torch.autocast(DEVICE, enabled=use_amp):
                e = bird_mae(x)[0]
            embs.append(e.float().cpu().numpy())
        t_gpu += time.time() - t
        clip_ids.extend(clips.tolist()); win_ids.extend(wins.tolist()); n_done += len(specs)
        if step % 100 == 0:
            done_clips = min(seen, len(files)); el = time.time() - t_start
            print(f"{done_clips}/{len(files)} clips | {n_done} windows | {el:.0f}s | "
                  f"ETA {el / done_clips * (len(files) - done_clips) / 60:.1f} min | model {t_gpu:.0f}s", flush=True)

    E = np.concatenate(embs).astype(np.float32)
    assert E.shape == (len(clip_ids), EMB_DIM), E.shape
    print(f"failed decodes: {len(failed)}", failed[:5])
    if len(failed) > max(3, 0.01 * len(files)):
        raise RuntimeError("Too many clips failed to decode")
    np.save(OUT / "birdmae_embeddings.tmp.npy", E)
    os.replace(OUT / "birdmae_embeddings.tmp.npy", OUT / "birdmae_embeddings.npy")
    pd.DataFrame({"clip": clip_ids, "win": win_ids}).to_csv(OUT / "birdmae_windows.csv", index=False)
    wall = time.time() - t_start
    json.dump(dict(fingerprint=fingerprint(), file_shas=file_shas, windows=len(E), clips=len(files), failed=failed,
                   wall_s=wall, model_s=t_gpu, windows_per_s=len(E) / wall),
              open(OUT / "birdmae_cache_meta.json", "w"), indent=1)
    print(f"TIMING | {len(E)} windows, {len(files)} clips | wall {wall:.0f}s | model {t_gpu:.0f}s | {len(E) / wall:.0f} windows/s")
    del bird_mae, embs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    cache = OUT
else:
    print("Reusing cached embeddings from", cache)

E = np.load(cache / "birdmae_embeddings.npy")
windows = pd.read_csv(cache / "birdmae_windows.csv")
file_shas = json.load(open(cache / "birdmae_cache_meta.json"))["file_shas"]
assert E.shape[1] == EMB_DIM and len(E) == len(windows)
print("embeddings", E.shape)

## Linear probe, trained from the cache only

In [ ]:
clip_of, win_of = windows["clip"].values, windows["win"].values
in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
assert not (in_train & in_val).any() and (in_train | in_val).all()

Y = np.zeros((len(windows), len(species)), np.float32)      # train: primary 1.0, secondary 0.5
for row, c in enumerate(clip_of):
    Y[row, sec_idx[c]] = CFG["secondary_weight"]
    Y[row, y_idx[c]] = 1.0
clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
W_row = (1.0 / clip_windows).astype(np.float32)             # each clip counts once

mean, std = E[in_train].mean(axis=0), E[in_train].std(axis=0) + 1e-6
Z = (E - mean) / std

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

# Validation rows, M1 protocol: first window, primary label only
rows_va = np.where(in_val & (win_of == 0))[0]
Y_va = np.zeros((len(rows_va), len(species)), np.float32)
Y_va[np.arange(len(rows_va)), y_idx[clip_of[rows_va]]] = 1.0

rows_all_va = np.where(in_val)[0]
va_groups = clip_of[rows_all_va]
order_clips = np.unique(va_groups)
Y_clip = np.zeros((len(order_clips), len(species)), np.float32)
Y_clip[np.arange(len(order_clips)), y_idx[order_clips]] = 1.0

def make_head(kind):
    assert kind == "linear"
    return nn.Sequential(nn.Linear(EMB_DIM, len(species)))

Z_dev = torch.from_numpy(Z).to(DEVICE); Y_dev = torch.from_numpy(Y).to(DEVICE); W_dev = torch.from_numpy(W_row).to(DEVICE)
tr_rows = torch.from_numpy(np.where(in_train)[0]).to(DEVICE)

@torch.no_grad()
def predict(model, rows):
    model.eval(); out = []
    for i in range(0, len(rows), 4096):
        out.append(torch.sigmoid(model(Z_dev[torch.from_numpy(rows[i:i + 4096]).to(DEVICE)])).cpu().numpy())
    return np.concatenate(out)

def evaluate(model):
    first, n_cls = macro_auc(Y_va, predict(model, rows_va))
    p = pd.DataFrame(predict(model, rows_all_va)).groupby(va_groups).max().loc[order_clips].values
    return first, macro_auc(Y_clip, p)[0], n_cls

def train_head(kind):
    torch.manual_seed(CFG["seed"])
    model = make_head(kind).to(DEVICE)
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=CFG["epochs"] * steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best = dict(first=-1.0)
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows), device=DEVICE)]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                bce = nn.functional.binary_cross_entropy_with_logits(model(Z_dev[b]).float(), Y_dev[b], reduction="none")
                loss = (bce.mean(1) * W_dev[b]).sum() / W_dev[b].sum()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()
        first, clip_max, n_cls = evaluate(model)
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max, epoch=ep, n_classes=n_cls,
                        state={k: v.cpu().clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        print(f"{kind:6s} ep {ep:2d} | loss {total / steps:.4f} | val AUC first-5s {first:.4f} | clip-max {clip_max:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    model.load_state_dict(best["state"])
    return model, best

results = {k: train_head(k) for k in ("linear",)}
print("\nHead   | val AUC first-5s (M1 protocol) | val AUC clip-max | best epoch | classes")
for k, (_, b) in results.items():
    print(f"{k:6s} | {b['first']:.4f} | {b['clip_max']:.4f} | {b['epoch']} | {b['n_classes']}")

## Save the probe and the species order

In [ ]:
best_kind = max(results, key=lambda k: results[k][1]["first"])
model, best = results[best_kind]
linears = [m for m in model if isinstance(m, nn.Linear)]
arrays = dict(mean=mean, std=std, n_layers=len(linears))
for i, m in enumerate(linears):
    arrays[f"W{i}"] = m.weight.detach().cpu().numpy(); arrays[f"b{i}"] = m.bias.detach().cpu().numpy()
np.savez(OUT / "birdmae_head.npz", **arrays)
json.dump(species, open(OUT / "species.json", "w"))

head = np.load(OUT / "birdmae_head.npz")       # inference path must reproduce the trained head
np_p = head_predict(head, E[rows_va])
assert np.allclose(np_p, predict(model, rows_va), atol=1e-4), "numpy head differs from torch head"

summary = dict(best_head=best_kind, val_auc_first5s=best["first"], val_auc_clip_max=best["clip_max"],
               val_classes=best["n_classes"], hf_repo=HF_REPO, hf_revision=HF_REVISION, birdmae_files_sha256=file_shas,
               transformers_pin=PINS, cfg=CFG,
               heads={k: {x: b[x] for x in ("first", "clip_max", "epoch")} for k, (_, b) in results.items()})
json.dump(summary, open(OUT / "head_cfg.json", "w"), indent=1)
print("best:", best_kind, f"val AUC {best['first']:.4f} | saved birdmae_head.npz, species.json, head_cfg.json")